# Chapter 2 — Calculating and Exploring Spectral Indices

This notebook calculates four vegetation and pigment-sensitive indices — **NDVI, CIRE, CRI2 and ARI2** — and explores them with index maps, histograms, an interactive index-to-index scatter plot, and a correlation matrix.

**Before starting:** keep this notebook beside `workshop_utils.py`, place the raster at the path reported by `DATA_PATH`, start with a fresh kernel and run the cells from top to bottom.

**Memory note:** index values, statistics, histograms and correlations use every valid pixel. Interactive maps are drawn from a display preview (at most `DISPLAY_MAX_SIDE` pixels on the longest side), and the scatter plot draws a random sample of pixels, so the browser stays responsive.

## 1. Load libraries and shared workshop functions

In [ ]:
from pathlib import Path
import sys
import rioxarray as rxr

import dask
import dask.array as dask_array
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
import ipywidgets as widgets

from IPython.display import display, Markdown, clear_output

pio.renderers.default = "plotly_mimetype+notebook_connected"

# Start a local Dask Client so workers and task activity are visible in the
# notebook. Threads avoid spawning extra Python processes and keep the worker
# setup reliable in Windows-based Jupyter environments.
from dask.distributed import Client, LocalCluster

DASK_WORKERS = 4
cluster = LocalCluster(
    n_workers=DASK_WORKERS,
    threads_per_worker=2,
    processes=False,
    dashboard_address=":0",
)
client = Client(cluster, set_as_default=True)
print(f"Dask scheduler: {client.scheduler.address}")
print(f"Dask workers: {len(client.scheduler_info()['workers'])}")
print(f"Dask dashboard: {client.dashboard_link}")
client

# The notebooks and workshop_utils.py should remain in the same folder.
NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "workshop_utils.py").exists():
    raise FileNotFoundError(
        "Open Jupyter in the converted workshop folder so workshop_utils.py "
        "and requirements.txt are beside this notebook."
    )
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from plotly.subplots import make_subplots
from workshop_utils import (
    BAND_INFO, pstretch, array_to_png_b64, to_uint8, WORKSHOP_INDICES, INDEX_INFO, 
    interactive_raster_explorer, plot_raster_distribution,
    projected_plot_context, make_projected_image, apply_projected_axes, compute_indices, plot_indices, save_indices_to_tiff, 
)

# Set the path to the GeoTIFF stored on your computer.
# Expected workshop layout: notebook_folder/data/your_raster.tif
RASTER_FILENAME = "NTABRT0007-20240828-ms-cog-subset3.tif"
RASTER_PATH = NOTEBOOK_DIR / "data" / RASTER_FILENAME

# _HERE = Path(__file__).resolve().parent
OUTPUT_DIR = NOTEBOOK_DIR / "outputs"

# Also allow the TIFF to sit beside the notebook.
if not RASTER_PATH.exists():
    local_file = NOTEBOOK_DIR / RASTER_FILENAME
    if local_file.exists():
        RASTER_PATH = local_file

if not RASTER_PATH.is_file():
    raise FileNotFoundError(
        f"Could not find the raster at {RASTER_PATH}. "
        "Edit RASTER_FILENAME / RASTER_PATH above to match the location "
        "of your local GeoTIFF."
    )

print(f"Local raster: {RASTER_PATH.resolve()}")
print(f"Dask scheduler: {client.scheduler.address}")
print(f"Dask workers: {len(client.scheduler_info()['workers'])}")


In [ ]:
# Open the GeoTIFF as lazy Dask chunks. Only chunks needed by a later
# calculation are read into memory. Keep one band per chunk to bound working RAM.
RASTER_CHUNKS = {"band": 1, "x": 512, "y": 512}
raster = rxr.open_rasterio(
    RASTER_PATH,
    masked=True,
    chunks=RASTER_CHUNKS,
    cache=False,
    lock=True,
).astype("float32")

expected_band_names = [band["var"] for band in BAND_INFO]
if raster.sizes.get("band") != len(expected_band_names):
    raster.close()
    raise ValueError(
        f"Expected {len(expected_band_names)} bands, found "
        f"{raster.sizes.get('band')} in {RASTER_PATH.name}."
    )

# The supplied image uses zero as background/NoData. Keep values lazy while
# masking those pixels and converting the band dimension to named variables.
raster = raster.assign_coords(band=expected_band_names)
raster = raster.where(np.isfinite(raster) & (raster > 0))
ds = raster.to_dataset(dim="band")
ds.attrs.update({
    "filename": RASTER_PATH.name,
    "source_path": str(RASTER_PATH.resolve()),
    "instrument": "MicaSense RedEdge-P Dual",
    "image_width_px": int(raster.sizes["x"]),
    "image_height_px": int(raster.sizes["y"]),
    "pixel_size_m": float(abs(raster.rio.resolution()[0])),
    "crs": str(raster.rio.crs),
    "transform": tuple(raster.rio.transform()),
    "bounds": tuple(float(v) for v in raster.rio.bounds()),
    "dask_chunks": str(RASTER_CHUNKS),
})
for band in BAND_INFO:
    ds[band["var"]].attrs.update({
        "long_name": band["name"],
        "centre_wavelength_nm": band["nm"],
        "fwhm_nm": band["fwhm"],
    })

ds


In [ ]:
# from pathlib import Path
# import sys
# import gc

# import numpy as np
# import pandas as pd
# import plotly.graph_objects as go
# import plotly.express as px
# import plotly.io as pio
# import ipywidgets as widgets
# from IPython.display import display, clear_output

# pio.renderers.default = "plotly_mimetype+notebook_connected"

# NOTEBOOK_DIR = Path.cwd()
# if not (NOTEBOOK_DIR / "workshop_utils.py").exists():
#     raise FileNotFoundError(
#         "Keep workshop_utils.py beside this notebook and start Jupyter "
#         "from the workshop folder."
#     )
# if str(NOTEBOOK_DIR) not in sys.path:
#     sys.path.insert(0, str(NOTEBOOK_DIR))

from workshop_utils import (
    INDEX_INFO,
    WORKSHOP_INDICES,
    # DATA_PATH,
    # OUTPUT_DIR,
    load_data,
    compute_indices,
    plot_indices,
    save_indices_to_tiff,
    interactive_raster_explorer,
)

# print(f"Raster path: {DATA_PATH}")

In [ ]:
invalid_mask = (
    ~np.isfinite(ds)
    | (ds == 0)
    | (ds < 0)
    | (ds > 1)
)

ds = ds.where(~invalid_mask)
ds

## 2. Index catalogue

| Index | Formula | Main interpretation |
|---|---|---|
| NDVI | (NIR-842 − Red-668) / (NIR-842 + Red-668) | General vegetation density and greenness |
| CIRE | (NIR-842 / Red-Edge-705) − 1 | Total chlorophyll via red-edge reflectance |
| CRI2 | (1 / Green-531) − (1 / Red-Edge-705) | Carotenoids, often elevated under stress or senescence |
| ARI2 | NIR-842 × [(1 / Green-560) − (1 / Red-Edge-705)] | Anthocyanin index weighted by NIR for canopy variation |

## 3. Load data and calculate indices

`compute_indices` adds the four indices to the dataset as new float32 variables. The original band arrays are shared, not copied.

In [ ]:
# ds = compute_indices(load_data(DATA_PATH))
ds = compute_indices(ds).compute()
print("Indices calculated:", ", ".join(WORKSHOP_INDICES))
print(f"Image: {ds.sizes['y']:,} rows × {ds.sizes['x']:,} columns")
print(f"Variables: {', '.join(ds.data_vars)}")
print(f"Dataset memory: {ds.nbytes / 1e6:,.0f} MB")

## 4. Index formulas and scene statistics

Statistics use all valid pixels. Percentiles are computed in a single pass per index.

In [ ]:
rows = []
for name in WORKSHOP_INDICES:
    values = ds[name].data
    valid = values[np.isfinite(values)]
    p2, median, p98 = np.percentile(valid, [2, 50, 98])
    rows.append({
        "Index": name,
        "Formula": INDEX_INFO[name]["formula"],
        "Min": float(valid.min()),
        "P2": float(p2),
        "Median": float(median),
        "Mean": float(valid.mean(dtype="float64")),
        "Std": float(valid.std(dtype="float64")),
        "P98": float(p98),
        "Max": float(valid.max()),
        "Valid px": int(valid.size),
    })
    del valid

index_stats = pd.DataFrame(rows)
display(index_stats.round(4))

## 5. Index maps

### 5.1 A single index

`plot_indices` returns a Plotly figure, so you can restyle it before showing it.

In [ ]:
fig_ndvi = plot_indices(
    ds,
    indices=["NDVI"],
    cols=1,
    robust=True,
    show_colorbars=False,
    show=False,
)
fig_ndvi.update_xaxes(title_font=dict(size=16), tickfont=dict(size=14))
fig_ndvi.update_yaxes(title_font=dict(size=16), tickfont=dict(size=14))
fig_ndvi.show(config={"scrollZoom": True, "displaylogo": False})
del fig_ndvi

### 5.2 All four indices

`robust=True` uses each index's P2–P98 range for the colour scale, so a few extreme pixels do not wash out the map.

In [ ]:
fig_all = plot_indices(
    ds,
    indices=["NDVI", "CIRE", "CRI2", "ARI2"],
    robust=True,
    show=False,
)
fig_all.update_xaxes(title_font=dict(size=14), tickfont=dict(size=11))
fig_all.update_yaxes(title_font=dict(size=14), tickfont=dict(size=11))
fig_all.update_layout(title_text="", margin=dict(t=60))
fig_all.show(config={"scrollZoom": True, "displaylogo": False})

# Optional: save the same figure as a stand-alone HTML file.
# fig_all.write_html(OUTPUT_DIR / "workshop_indices.html", include_plotlyjs="cdn")
del fig_all
# gc.collect()

## 6. Save the indices as GeoTIFFs

Exports are full resolution and keep the source CRS and geotransform. The first call writes one multi-band file; the second writes one file per index.

In [ ]:
save_indices_to_tiff(ds, out_path=OUTPUT_DIR / "workshop_indices.tif")
save_indices_to_tiff(ds, indices=["NDVI", "CIRE", "ARI2"], out_dir=OUTPUT_DIR / "indices")

## 7. Interactive index explorer

Choose an index to redraw its map and histogram. The histogram and statistic lines use every valid pixel.

### 7.1 Full data range (min, median, max)

In [ ]:
index_plot_info = {
    name: {
        **INDEX_INFO[name],
        "label": f"{name} — {INDEX_INFO[name]['full_name']}",
        "dropdown_label": name,
        "colorscale": INDEX_INFO[name]["cmap"],
        "histogram_color": "#397d54",
        "value_label": name,
    }
    for name in WORKSHOP_INDICES
}

index_selector_full, index_output_full = interactive_raster_explorer(
    ds,
    variables=WORKSHOP_INDICES,
    metadata=index_plot_info,
    default="NDVI",
    dropdown_description="Index:",
    robust=False,
    bins=100,
    statistics=("min", "median", "max"),
    show_colorbar=True,
    width=1150,
    height=550,
)

### 7.2 Robust colour stretch with index descriptions

Compare this view with 7.1: the P2–P98 stretch reveals spatial detail that a full-range colour scale hides when a few pixels are extreme.

In [ ]:
index_selector_robust, index_output_robust = interactive_raster_explorer(
    ds,
    variables=WORKSHOP_INDICES,
    metadata=index_plot_info,
    default="NDVI",
    dropdown_description="Index:",
    robust=True,
    percentiles=(2, 98),
    bins=90,
    statistics=("mean", "p2", "p98"),
    show_colorbar=True,
    show_metadata=True,
    width=1150,
    height=550,
)

## 8. Interactive index-to-index scatter plot

Drawing every pixel (millions of points) would freeze the browser, so the plot shows a random sample. Use the slider to change the sample size; the correlation coefficient in the title uses **all** valid pixel pairs.

In [ ]:
sc_x = widgets.Dropdown(options=WORKSHOP_INDICES, value="NDVI", description="X:")
sc_y = widgets.Dropdown(options=WORKSHOP_INDICES, value="CIRE", description="Y:")
sc_n = widgets.IntSlider(
    value=20_000, min=5_000, max=100_000, step=5_000,
    description="Samples:", continuous_update=False,
)
scatter_output = widgets.Output()
rng = np.random.default_rng(42)


def render_scatter(change=None):
    with scatter_output:
        clear_output(wait=True)
        x_name, y_name = sc_x.value, sc_y.value
        x_all = ds[x_name].data.ravel()  # views, no copies
        y_all = ds[y_name].data.ravel()
        valid_idx = np.flatnonzero(np.isfinite(x_all) & np.isfinite(y_all))
        if valid_idx.size < 2:
            print("Not enough valid pixel pairs.")
            return
        r = float(np.corrcoef(x_all[valid_idx], y_all[valid_idx])[0, 1])
        n = min(sc_n.value, valid_idx.size)
        sample = np.sort(rng.choice(valid_idx, size=n, replace=False))

        fig = go.Figure(go.Scattergl(
            x=x_all[sample], y=y_all[sample], mode="markers",
            marker=dict(size=3, opacity=0.35, color="#2c7fb8"),
            showlegend=False,
            hovertemplate=f"{x_name}: %{{x:.4f}}<br>{y_name}: %{{y:.4f}}<extra></extra>",
        ))
        fig.update_layout(
            title=(f"{x_name} vs {y_name} — {n:,} sampled of {valid_idx.size:,} valid pixels "
                   f"(Pearson r = {r:.3f}, all pixels)"),
            xaxis_title=x_name, yaxis_title=y_name,
            template="plotly_white", height=520,
            margin=dict(l=70, r=30, t=70, b=60),
        )
        fig.show(config={"displaylogo": False})
        del fig, valid_idx, sample


for control in (sc_x, sc_y, sc_n):
    control.observe(render_scatter, names="value")
display(widgets.VBox([widgets.HBox([sc_x, sc_y]), sc_n, scatter_output]))
render_scatter()

## 9. Correlation between indices

Pearson correlations use every pixel that is valid in all four indices. The matrix is a small 4 × 4 table, so it is cheap to display.

In [ ]:
complete_stack = np.column_stack([ds[name].data.ravel() for name in WORKSHOP_INDICES])
complete_stack = complete_stack[np.all(np.isfinite(complete_stack), axis=1)]
corr_matrix = pd.DataFrame(
    np.corrcoef(complete_stack, rowvar=False),
    index=WORKSHOP_INDICES,
    columns=WORKSHOP_INDICES,
)

fig_corr = px.imshow(
    corr_matrix.round(3),
    text_auto=".2f",
    color_continuous_scale="RdYlGn",
    zmin=-1,
    zmax=1,
    aspect="equal",
)
fig_corr.update_layout(
    title=f"Pearson correlation matrix — n={complete_stack.shape[0]:,} complete pixels",
    height=480,
    width=560,
    template="plotly_white",
)
fig_corr.show(config={"displaylogo": False})
del fig_corr

### 9.1 Correlation matrix with distributions

The lower triangle shows the correlations; the diagonal shows each index's smoothed distribution. The KDE uses a random sample of 50,000 pixels for speed — this affects only the drawn curve, not the correlations.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import gaussian_kde

values = corr_matrix.to_numpy()
labels = corr_matrix.columns
n = len(labels)

mask = np.triu(np.ones_like(values, dtype=bool), k=0)  # hide diagonal + upper triangle
corr_cmap = LinearSegmentedColormap.from_list("correlation", ["#5b8fa8", "#f2f2f2", "#c95735"])

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(np.ma.array(values, mask=mask), cmap=corr_cmap, vmin=-1, vmax=1, interpolation="none")
ax.set_xticks(np.arange(n))
ax.set_yticks(np.arange(n))
ax.set_xticklabels(labels, rotation=90)
ax.set_yticklabels(labels)

for i in range(n):
    for j in range(i):
        ax.text(j, i, f"{values[i, j]:.2f}", ha="center", va="center", fontsize=10, color="black")

kde_rng = np.random.default_rng(42)
for i in range(n):
    data = complete_stack[:, i]
    if data.size > 50_000:
        data = kde_rng.choice(data, size=50_000, replace=False)
    kde_ax = ax.inset_axes([i - 0.45, i - 0.45, 0.9, 0.9], transform=ax.transData, zorder=3)
    if data.size > 1 and np.ptp(data) > 0:
        low, high = np.percentile(data, [1, 99])
        x_grid = np.linspace(low, high, 200)
        density = gaussian_kde(data[(data >= low) & (data <= high)])(x_grid)
        kde_ax.fill_between(x_grid, density, color="#5b8fa8", alpha=0.35)
        kde_ax.plot(x_grid, density, color="#5b8fa8", linewidth=1.2)
    kde_ax.set_facecolor("white")
    kde_ax.set_xticks([])
    kde_ax.set_yticks([])
    for spine in kde_ax.spines.values():
        spine.set_visible(False)

ax.set_xticks(np.arange(-0.5, n, 1), minor=True)
ax.set_yticks(np.arange(-0.5, n, 1), minor=True)
ax.grid(which="minor", color="white", linewidth=1)
ax.tick_params(which="minor", bottom=False, left=False)
ax.set_title("Correlation matrix")
plt.tight_layout()
plt.show()
plt.close(fig)

del complete_stack
# gc.collect()

## Activities

1. Compare NDVI and CIRE. Can you find dense-canopy areas where CIRE retains variation but NDVI appears saturated?
2. Locate the highest-ARI2 cluster, then compare the same location in NDVI.
3. Identify the strongest positive and the weakest/most negative pairs in the correlation matrix. Which pigment proxy is least related to NDVI?

ARI2 and CRI2 can be negative where the reciprocal-difference formulas are applied to soil or shadow pixels. The scatter plot samples pixels to keep the browser responsive; its correlation value uses all pixels.